In [0]:
#| default_exp review

## Reviewing notebooks

Diff and style feedback that focuses on code cells.

In [ ]:
#| export
import subprocess
from pathlib import Path

from chkstyle.core import main as _chkstyle_main
from fastcore.script import Param, call_parse
from nbdev.diff import nbs_pair, source_diff

from nbskill.foundation import _cli_error, _cli_return, _none_if_string, _tracked_call

In [ ]:
#| export
def _chstyle_argv(path=".", skip_folder_re=None, skip_path=None):
    path = "." if path is None else str(path)
    argv = ["chstyle", path]
    if skip_folder_re: argv += ["--skip-folder-re", str(skip_folder_re)]
    if skip_path: argv += ["--skip-path", str(skip_path)]
    return argv

In [ ]:
#| export
def _run_chstyle(path=".", skip_folder_re=None, skip_path=None, strict=False):
    status = _chkstyle_main(_chstyle_argv(path, skip_folder_re, skip_path))
    if strict and status: raise SystemExit(status)
    return status

In [ ]:
#| export
@call_parse
@_tracked_call
def chstyle(
    path: Param("File or folder to check", str, opt=False, nargs="?") = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Folder name/path to skip
    strict: bool = False,  # Exit non-zero when style hints are found
):
    "Print fast.ai style hints using fastaistyle/chkstyle."
    status = _run_chstyle(path, skip_folder_re, skip_path, strict)
    return _cli_return(status)

In [ ]:
#| export
def code_source(cell): return cell.source if cell.cell_type == "code" else None

In [ ]:
#| export
def _git_ref_path_error(path, ref):
    if ref is None: return None
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0:
        return f"No git repository found for {str(path)!r}."
    root = Path(root_cmd.stdout.strip())
    try:
        rel = path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError:
        return f"{str(path)!r} is outside git repository {str(root)!r}."
    spec = f"{ref}:{rel}"
    exists_cmd = subprocess.run(
        ["git", "-C", str(root), "cat-file", "-e", spec],
        capture_output=True, text=True,
    )
    if exists_cmd.returncode == 0: return None
    return (
        f"Could not find notebook {rel!r} at git ref {ref!r}. "
        "The notebook may be new relative to that ref, or the repository may not have a HEAD commit yet. "
        "Commit the notebook first, choose an existing ref/path, or pass --ref_a None to compare against the working tree."
    )


@call_parse
@_tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str|None = "HEAD",  # First git ref; use None for working tree
    ref_b: str|None = None,  # Second git ref; defaults to working tree
    adds: bool = True,  # Include code cells added in ref_b
    changes: bool = True,  # Include changed code cells
    dels: bool = False,  # Include deleted code cells
):
    "Print nbdev-style diffs for code cells only."
    ref_a, ref_b = _none_if_string(ref_a), _none_if_string(ref_b)
    if msg := (_git_ref_path_error(path, ref_a) or _git_ref_path_error(path, ref_b)):
        _cli_error(msg)
    try:
        old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    except Exception as exc:
        detail = str(exc)
        hint = (
            f"Could not diff {path!r} against {ref_a!r}. "
            "The notebook may be new relative to that git ref, or the repository may not have a HEAD commit yet. "
            "Commit the notebook first, or pass --ref_a None to compare against the working tree."
        )
        if detail: hint += f"\nUnderlying error: {detail}"
        _cli_error(hint)
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    blocks = []
    if adds:    blocks += [(cid, source_diff("", new[cid])) for cid in new if cid not in old]
    if changes: blocks += [(cid, source_diff(old[cid], new[cid])) for cid in new if cid in old and new[cid] != old[cid]]
    if dels:    blocks += [(cid, source_diff(old[cid], "")) for cid in old if cid not in new]
    text = "\n\n".join(f"--- code cell {cid} ---\n{diff}" for cid, diff in blocks if diff.strip())
    if text: print(text)
    else: print("No code cell changes")
    return _cli_return(text)

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.review import diff_nb

with _tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "demo.ipynb"
    _write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    try:
        diff_nb(str(path))
    except SystemExit as exc:
        assert exc.code == 1

In [0]:
from fastcore.nbio import mk_cell
from nbskill.review import code_source

cell = mk_cell("x = 1", cell_type="code")
assert code_source(cell) == "x = 1"